## Conducting Topic Modeling with BERTopic
### Rating: X

This Colab conducts topic modeling with BERTopic at the bucket level. So, for each level of violence, the corpora for each game is combined under their rating label (E 10+, T, M), and passed through this BERTopic pipeline.

In [1]:
import pandas as pd

### BERTopic Pipeline

In [ ]:
rating_bucket = "M"

corpus_convo_df = pd.read_csv(f"combined_convo_sent_{rating_bucket}_avg.csv")

rating_bucket_title = "Mature"

game_type = f"{rating_bucket_title} Rated Game Corpus"

In [ ]:
!pip install bertopic sentence-transformers

In [ ]:
# Filters out posts less than 300 characters, or ~50 words, this is done to ensure that 
# overly short threads are filtered out as BERTopic does not struggle with little context
# or, create far too many topics

print(corpus_convo_df.shape)

corpus_convo_df = corpus_convo_df[corpus_convo_df['text'].str.len() > 300]

print(corpus_convo_df.shape)

posts = corpus_convo_df['text'].to_list()

corpus_convo_df

In [ ]:
from sentence_transformers import SentenceTransformer

# Pre-calculate embeddings for our posts, doing this once ensures that
# we don't need to recalculate the embeddings for each run 

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2",
    device="cuda"
)

embeddings = embedding_model.encode(posts, show_progress_bar=True)

In [ ]:
from umap import UMAP

# This is the reduction method we are using, we explicity set the 
# random state for reproducable results, but there still is some
# variation in the final output that I couldn't control for

umap_model = UMAP(random_state=42)

In [ ]:
from hdbscan import HDBSCAN

# This is the clustering method used for grouping the reduced embeddings

hdbscan_model = HDBSCAN(min_cluster_size=9, prediction_data=True)

In [ ]:
import nltk
from sklearn.feature_extraction.text import CountVectorizer
from nltk.corpus import stopwords
nltk.download('stopwords')


# Game specific stopwords
game_keywords = ["destiny", "destiny2", "bf","d1", "d2", "battlefront", "plantsvszombies", "pvz",
            "doom", "cod", "blackops", "gardenwarfare", "doom", "bo3"]

# Common words that were frequently appearing in topics that were not informative
uninformative_words = ["game", "games", "playing", "play", "plays", "people", "think",
                       "would", "like", "could"]

# The official ntlk list of stopwords
stop_words = (stopwords.words('english'))

stop_words = (stop_words + game_keywords + uninformative_words)

# This is the tokenizer that removes stops words and ensures unigram and bigrams 
# appear in the topic representations
vectorizer_model = CountVectorizer(stop_words=stop_words, min_df=2, ngram_range=(1, 2))


In [ ]:
from bertopic.representation import KeyBERTInspired

# Representation model outlined in the BERTopic pipeline
keybert_model = KeyBERTInspired()

In [ ]:
# Final BERTopic pipeline

from bertopic import BERTopic

topic_model_convo = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,

    representation_model=keybert_model,

    language="english",

    calculate_probabilities=True,
    verbose=True
)

topics, probs = topic_model_convo.fit_transform(posts, embeddings)

### Reviewing Results
This section goes through the three reduction steps, looking first at the original output of topics, which are saved, the semi-reduced topics, which were used with LLM prompting for simplifying the results, and the heavily reduced topics. 

### No Reduction and Visualization

In [ ]:
# Shows the original topics without the representative_Docs
df_topics_og = topic_model_convo.get_topic_info()
df_topics_og.drop('Representative_Docs', axis=1)

In [ ]:
# Reduce dimensionality of embeddings for ploting
reduced_embeddings = UMAP(n_neighbors=10, n_components=2, min_dist=0.0, metric='cosine').fit_transform(embeddings)

topic_model_convo.visualize_documents(posts, reduced_embeddings=reduced_embeddings)

In [ ]:
topic_model_convo.visualize_heatmap()

In [ ]:
topic_model_convo.visualize_topics()

In [ ]:
topic_model_convo.visualize_barchart(top_n_topics=10, n_words=8)

In [ ]:
topic_model_convo.visualize_hierarchy()

### Semi-Reduction, LLM/Manual Labels, Visualizations

In [ ]:
# Reduce the number of topics to 100
topic_reduced = topic_model_convo.reduce_topics(nr_topics=100, docs=posts)

In [ ]:
# Reapplies our vectorizer model after reducing the topics
topic_reduced.update_topics(posts, vectorizer_model=vectorizer_model)

In [ ]:
# Shows the reduced topics without the representative_Docs
df_topics_red = topic_reduced.get_topic_info()
df_topics_red.drop('Representative_Docs', axis=1)

In [ ]:
doc_topic_df = corpus_convo_df.copy()

# Add label indices for each row in the dataset from the reduced topics
doc_topic_df["topic_index_label"] = topic_reduced.topics_

# Paste in LLM/Manually Coded Topic Labels and their corresponding indices, bellow is a sample from 
# the M rated corpus

base_labels = {
    "topic_index": [
        -1, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9,
        10, 11, 12, 13, 14, 15, 16, 17, 18, 19,
        20, 21, 22, 23, 24, 25, 26, 27, 28, 29,
        30, 31, 32, 33, 34, 35, 36, 37, 38, 39,
        40, 41, 42, 43, 44, 45, 46, 47, 48, 49,
        50, 51, 52, 53, 54, 55, 56, 57, 58, 59,
        60, 61, 62, 63, 64, 65, 66, 67, 68, 69,
        70, 71, 72, 73, 74, 75, 76, 77, 78, 79,
        80, 81, 82, 83, 84, 85, 86, 87, 88, 89,
        90, 91, 92, 93, 94, 95, 96, 97, 98
    ],
    "base_top_label": [
        "Other", "Gameplay", "Gameplay", "Other", "Gameplay", "Gameplay", "Gameplay", "Social", "Gameplay", "Gameplay", "Customization",
        "Gameplay", "Gameplay", "Problems", "Customization", "Gameplay", "Gameplay", "Gameplay", "Gameplay", "Gameplay", "Gameplay",
        "Hardware", "Gameplay", "Gameplay", "Gameplay", "Gameplay", "Other", "Gameplay", "Gameplay", "Problems", "Customization",
        "Problems", "Other", "Gameplay", "Social", "Gameplay", "Other", "Gameplay", "Gameplay", "Gameplay", "Gameplay",
        "Gameplay", "Gameplay", "Gameplay", "Social", "Gameplay", "Customization", "Other", "Hardware", "Hardware", "Gameplay",
        "Hardware", "Gameplay", "Gameplay", "Customization", "Gameplay", "Other", "Other", "Gameplay", "Gameplay", "Gameplay",
        "Gameplay", "Gameplay", "Gameplay", "Hardware", "Social", "Customization", "Gameplay", "Gameplay", "Gameplay", "Social",
        "Gameplay", "Gameplay", "Social", "Gameplay", "Problems", "Other", "Customization", "Gameplay", "Social", "Gameplay",
        "Gameplay", "Gameplay", "Social", "Gameplay", "Gameplay", "Gameplay", "Other", "Social", "Gameplay", "Gameplay",
        "Gameplay", "Other", "Other", "Gameplay", "Gameplay", "Gameplay", "Gameplay", "Gameplay", "Other"
    ]
}

# Collect BERTopic coded topic labels and their corresponding indices
topic_info = topic_model_convo.get_topic_info()

# Create final dataframe maping the simplified topics to their corresponding indecies 

topic_labels_com = dict(zip(topic_info.Topic, topic_info.Name))
topic_labels_base = dict(zip(base_labels["topic_index"], base_labels["base_top_label"]))

doc_topic_df["high_top_label"] = doc_topic_df["topic_index_label"].map(topic_labels_com)
doc_topic_df["topic_label"] = doc_topic_df["topic_index_label"].map(topic_labels_base)

In [ ]:
import csv
doc_topic_df.to_csv("final_topic_convo_M.csv",index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)
topic_info.drop('Representative_Docs', axis=1).to_csv("final_topic_labels_M.csv",index=False, encoding="utf-8", quoting=csv.QUOTE_ALL)

In [ ]:
topic_model_convo.visualize_documents(posts, reduced_embeddings=reduced_embeddings)

In [ ]:
topic_reduced.visualize_heatmap()

In [ ]:
topic_model_convo.visualize_topics()

In [ ]:
topic_model_convo.visualize_barchart(top_n_topics=10, n_words=8)

In [ ]:
topic_model_convo.visualize_hierarchy()

### Extreme Reduction

This reduces the number of topics to just 20, and was mostly done as a test to see what would happen. Discussed further in the study, but ultimately overgeneralizes topics. 

In [ ]:
# Reduce the number of topics to 20
topic_reduced = topic_model_convo.reduce_topics(nr_topics=20, docs=posts)

In [ ]:
topic_reduced.update_topics(posts, vectorizer_model=vectorizer_model)

In [ ]:
df_topics_red = topic_reduced.get_topic_info()
df_topics_red.drop('Representative_Docs', axis=1)

In [ ]:
topic_model_convo.visualize_documents(posts, reduced_embeddings=reduced_embeddings)

In [ ]:
topic_model_convo.visualize_heatmap()

In [ ]:
topic_model_convo.visualize_topics()

In [ ]:
topic_model_convo.visualize_barchart(top_n_topics=10, )

In [ ]:
topic_model_convo.visualize_hierarchy()